# task_09 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

campaigns = pd.read_csv(base / "campaigns.csv")
ad_spend = pd.read_csv(base / "ad_spend.csv")
sessions = pd.read_csv(base / "sessions.csv")
orders = pd.read_csv(base / "orders.csv")

sessions["session_date"] = pd.to_datetime(sessions["session_date"])
ad_spend["date"] = pd.to_datetime(ad_spend["date"])

sessions_orders = sessions.merge(orders, on="session_id").merge(campaigns, on="campaign_id")
revenue_by_channel = sessions_orders.groupby("channel")["order_value"].sum()
spend_by_channel = ad_spend.merge(campaigns, on="campaign_id").groupby("channel")["spend"].sum()
q1 = str((revenue_by_channel / spend_by_channel).sort_values(ascending=False).index[0])

search_sessions = sessions.merge(campaigns[["campaign_id", "channel"]], on="campaign_id")
search_sessions = search_sessions[search_sessions["channel"] == "search"]
bounce_rate = search_sessions.groupby("session_date").apply(lambda g: (g["pages_viewed"] == 1).mean(), include_groups=False)
q2 = bounce_rate[bounce_rate == bounce_rate.max()].index.min().date().isoformat()

first_sessions = sessions.merge(campaigns[["campaign_id", "channel"]], on="campaign_id").sort_values(["user_id", "session_date", "session_id"]).groupby("user_id").first()
social_users = set(first_sessions[first_sessions["channel"] == "social"].index)
social_orders = sessions[sessions["user_id"].isin(social_users)].merge(orders, on="session_id")
q3 = str(int(social_orders["order_value"].median()))

sessions_with_channel = sessions.merge(campaigns[["campaign_id", "channel"]], on="campaign_id")
purchase_sessions = sessions.merge(orders[["session_id"]], on="session_id")
first_purchase_date = purchase_sessions.groupby("user_id")["session_date"].min()
users_multi_channel = 0
for user_id, first_date in first_purchase_date.items():
    channels_before = sessions_with_channel[(sessions_with_channel["user_id"] == user_id) & (sessions_with_channel["session_date"] < first_date)]["channel"].nunique()
    if channels_before >= 2:
        users_multi_channel += 1
q4 = str(users_multi_channel)

revenue_by_campaign = sessions.merge(orders, on="session_id").groupby("campaign_id")["order_value"].sum()
non_bounce_sessions = sessions[sessions["pages_viewed"] > 1].groupby("campaign_id").size()
q5 = str((revenue_by_campaign / non_bounce_sessions).dropna().sort_values(ascending=False).index[0])


Q1: Using revenue attributed only to orders linked to sessions in this dataset, which channel has the highest ROAS defined as total order_value divided by total spend?

In [ ]:
print(q1)


Q2: Among search-channel sessions, which is the earliest session_date among the date or dates with the highest bounce rate, where bounce means pages_viewed = 1?

In [ ]:
print(q2)


Q3: What is the median order_value for users whose first recorded session came from the social channel?

In [ ]:
print(q3)


Q4: How many users had sessions from at least two distinct channels before their first purchase?

In [ ]:
print(q4)


Q5: Which campaign_id has the highest revenue per non-bounce session, where non-bounce means pages_viewed > 1?

In [ ]:
print(q5)
